# 03 Confidentiality and Opacity

This notebook studies the `CONFIDENCIAL` records in the RNPDNO mirror.

Working question:

> When does confidentiality protect people, and when does large-scale redaction become a public accountability problem?

ADHD scope rule for this notebook: three charts, one interpretation table, one next-step decision. Stop there.

## What We Know Going In

Confidentiality can be legitimate. The public version of a disappearance registry should not expose sensitive personal data, active search details, children, families, witnesses, or information that could endanger people.

But mass redaction also has consequences. If a record is `CONFIDENCIAL`, this local file keeps the state and report origin, but hides the status, dates, sex, birth date, and municipality. That means the person still counts in the unresolved stock, but many analyses become impossible.

Our standard sentence:

> Confidentiality may protect people at the individual level, while mass redaction can weaken accountability at the system level.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CSV = ROOT / "data" / "external" / "data.csv"
FIGDIR = ROOT / "reports" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV, dtype="string")
df["is_confidential"] = df["ESTATUS_VICTIMA"].eq("CONFIDENCIAL")

INK = "#2b2b2b"
RED = "#7a1f1f"
MUTED = "#8b8b8b"
BLUE = "#3b5b7a"

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": MUTED,
    "axes.titleweight": "bold",
    "axes.titlecolor": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
})

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

## 1. How Big Is `CONFIDENCIAL`?

First, do not interpret. Count.

In [ ]:
status = (
    df["ESTATUS_VICTIMA"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="records")
)
status["share"] = status["records"] / len(df)
status

In [ ]:
plot_status = status.sort_values("records")

fig, ax = plt.subplots(figsize=(7, 3.8))
colors = [RED if s == "CONFIDENCIAL" else INK for s in plot_status["status"]]
ax.barh(plot_status["status"], plot_status["records"], color=colors)
ax.set_title("Status mix in the local RNPDNO mirror")
ax.set_xlabel("Records")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

for i, row in enumerate(plot_status.itertuples(index=False)):
    ax.text(row.records, i, f" {row.records:,.0f} ({row.share:.1%})", va="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIGDIR / "08_status_mix_confidential.png", bbox_inches="tight")
plt.show()

## 2. What Does `CONFIDENCIAL` Hide?

A confidential record is not useless, but it is limited. This table checks field availability by status.

In [ ]:
fields = [
    "ID_VICTIMA",
    "ORIGEN_REPORTE",
    "FECHA_NACIMIENTO",
    "SEXO",
    "FECHA_DESAPARICION",
    "FECHA_REGISTRO",
    "ESTATUS_VICTIMA",
    "ENTIDAD",
    "MUNICIPIO",
]

availability = []
for status_value, sub in df.groupby("ESTATUS_VICTIMA", dropna=False):
    for field in fields:
        missing = sub[field].isna() | sub[field].eq("") | sub[field].eq("CONFIDENCIAL")
        availability.append({
            "status": status_value,
            "field": field,
            "available_share": round((~missing).mean(), 3),
        })

availability_table = (
    pd.DataFrame(availability)
    .pivot(index="field", columns="status", values="available_share")
    .reset_index()
)
availability_table

Interpretation:

- `CONFIDENCIAL` can still support state-level unresolved-stock counts.
- `CONFIDENCIAL` cannot support date, age, sex, municipality, or legal-category analysis.
- Therefore, any chart by year, age, sex, or municipality uses a smaller and potentially biased subset.

## 3. Where Is Opacity Concentrated?

A state with many confidential records is not necessarily hiding something intentionally. But the effect is still real: less public information is available for that state.

In [ ]:
state_opacity = (
    df.groupby("ENTIDAD", dropna=False)
    .agg(
        total_records=("ID_VICTIMA", "size"),
        confidential_records=("is_confidential", "sum"),
    )
    .reset_index()
)
state_opacity["confidential_share"] = state_opacity["confidential_records"] / state_opacity["total_records"]

state_opacity.sort_values("confidential_share", ascending=False).head(12)

In [ ]:
min_records = 1000
top_share = (
    state_opacity[state_opacity["total_records"] >= min_records]
    .sort_values("confidential_share", ascending=False)
    .head(15)
    .sort_values("confidential_share")
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_share["ENTIDAD"], top_share["confidential_share"], color=RED)
ax.set_title("States with the highest confidential-record share")
ax.set_xlabel("Share of records marked CONFIDENCIAL")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

national_share = df["is_confidential"].mean()
ax.axvline(national_share, color=INK, linestyle="--", linewidth=1)
ax.text(national_share, -0.75, f" national {national_share:.0%}", fontsize=8)

for i, row in enumerate(top_share.itertuples(index=False)):
    ax.text(row.confidential_share, i, f" {row.confidential_share:.0%}", va="center", fontsize=8)

ax.text(
    0,
    -0.14,
    f"Only states with at least {min_records:,} total records. This measures public-data opacity, not proven intent.",
    transform=ax.transAxes,
    fontsize=8,
    color=MUTED,
)

fig.tight_layout()
fig.savefig(FIGDIR / "09_confidential_share_by_state.png", bbox_inches="tight")
plt.show()

## 4. Count vs. Share

Two states can both matter for different reasons:

- high confidential count: many people are hidden from detailed analysis
- high confidential share: a large portion of that state's public data is opaque

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    state_opacity["total_records"],
    state_opacity["confidential_share"],
    s=(state_opacity["confidential_records"] ** 0.5) * 3,
    color=BLUE,
    alpha=0.75,
    edgecolor="white",
    linewidth=0.6,
)
ax.set_title("Opacity has both scale and intensity")
ax.set_xlabel("Total unresolved records by state")
ax.set_ylabel("Share marked CONFIDENCIAL")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

label_states = set(
    state_opacity.nlargest(5, "confidential_records")["ENTIDAD"]
).union(state_opacity.nlargest(5, "confidential_share")["ENTIDAD"])

for row in state_opacity[state_opacity["ENTIDAD"].isin(label_states)].itertuples(index=False):
    ax.text(row.total_records, row.confidential_share, f" {row.ENTIDAD}", fontsize=7)

ax.text(
    0,
    -0.16,
    "Bubble size scales with confidential-record count. Labels mark the top states by count or share.",
    transform=ax.transAxes,
    fontsize=8,
    color=MUTED,
)

fig.tight_layout()
fig.savefig(FIGDIR / "10_confidential_scale_vs_intensity.png", bbox_inches="tight")
plt.show()

## 5. What Can We Still Use?

| Question | Can confidential rows help? | Why |
|---|---|---|
| How many unresolved records exist by state? | Yes | `ENTIDAD` remains available. |
| Which states have more public-data opacity? | Yes | `CONFIDENCIAL` itself is the signal. |
| How has disappearance changed by year? | No | disappearance date is hidden. |
| Who is affected by sex or age? | No | sex and birth date are hidden. |
| Which municipalities are most affected? | No | municipality is hidden. |
| Are these crime-linked disappearances or no-localizadas? | No | status is hidden beyond the redaction marker. |

## 6. Should We Bring In Other Sources Now?

Yes, but not chaotically. The next external-data step should answer a specific gap exposed here.

Recommended order:

1. Finish this opacity profile inside the RNPDNO mirror.
2. Compare state totals against the live RNPDNO dashboard or a Data Civica version.
3. Bring in clandestine grave data for a separate layer, not as a replacement for missing-person records.
4. Build a madres buscadoras event table only after we define fields and source-confidence rules.

Do not use outside sources just because we distrust the government data. Use them to test specific claims and document specific blind spots.

## Exit Ticket

Write short answers:

1. What is the strongest fair claim we can make about `CONFIDENCIAL`?
2. What claim would go too far without more evidence?
3. Which state should we investigate first, and why: high count, high share, or both?
4. What external source would best test the pattern we see here?
5. What chart from this notebook belongs in the first public draft?